In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import sparse
import sklearn
import altair as alt

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer, make_column_selector, make_column_transformer
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from BorutaShap import BorutaShap

import sys
sys.path.append('../scripts')  # Go up one level to access scripts/
from data_cleaner import DataCleaner
from data_cleaner import CleanerConfig

In [ ]:
df = pd.read_csv('../data/inegi.csv',  low_memory=False)

# 1: Feature selection
df.head()
df.describe()

,ENTIDAD,NOM_ENT,MUN,NOM_MUN,LOC,NOM_LOC,LONGITUD,LATITUD,ALTITUD,POBTOT,...,VPH_CEL,VPH_INTER,VPH_STVP,VPH_SPMVPI,VPH_CVJ,VPH_SINRTV,VPH_SINLTC,VPH_SINCINT,VPH_SINTIC,TAMLOC
0,1,Aguascalientes,0,Total de la entidad Aguascalientes,0,Total de la Entidad,NaN,NaN,NaN,1425607,...,359895,236003,174089,98724,70126,6021,15323,128996,1711,*
1,1,Aguascalientes,0,Total de la entidad Aguascalientes,9998,Localidades de una vivienda,NaN,NaN,NaN,3697,...,732,205,212,48,41,39,62,530,20,*
2,1,Aguascalientes,0,Total de la entidad Aguascalientes,9999,Localidades de dos viviendas,NaN,NaN,NaN,3021,...,470,146,156,35,38,25,44,330,11,*
3,1,Aguascalientes,1,Aguascalientes,0,Total del Municipio,NaN,NaN,NaN,948990,...,251719,178619,130290,80951,56131,3299,7293,74227,731,*
4,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,"102°17'45.768"" W","21°52'47.362"" N",1878.0,863893,...,232793,169675,123670,77719,53589,2995,5984,63661,595,13


In [37]:
# ---------------------------------
# 1. Símbolos comunes en datos censales
# ---------------------------------
valores_no_disponibles = [
    "*", "N/D", "ND", "NA", "nan", ".", "", " "
]

# ---------------------------------
# 2. Función de limpieza universal
# ---------------------------------
def limpiar_columna_numerica(serie):
    serie = serie.astype(str)

    # eliminar espacios
    serie = serie.str.strip()

    # normalizar valores faltantes
    serie = serie.replace(valores_no_disponibles, np.nan)

    # eliminar separadores de miles
    serie = serie.str.replace(",", "", regex=False)

    # convertir a número
    serie = pd.to_numeric(serie, errors="coerce")

    return serie

# ---------------------------------
# 3. Aplicar limpieza a TODAS las columnas no categóricas
# ---------------------------------
columnas_limpiadas = []
reporte_limpieza = []

for col in df.columns:
    antes_na = df[col].isna().sum()
    tipo_original = df[col].dtype

    serie_limpia = limpiar_columna_numerica(df[col])

    # medir si realmente cambió algo
    despues_na = serie_limpia.isna().sum()
    cambios = (serie_limpia.notna() != df[col].notna()).sum()

    # decidir si la columna es numérica válida
    if serie_limpia.notna().sum() > 0:
        df[col] = serie_limpia
        columnas_limpiadas.append(col)

        reporte_limpieza.append({
            "variable": col,
            "tipo_original": str(tipo_original),
            "valores_convertidos": int(cambios),
            "nulos_finales": int(despues_na)
        })

reporte_limpieza = pd.DataFrame(reporte_limpieza)

print("\n=== LIMPIEZA COMPLETADA ===")
print("Columnas convertidas a numérico:", len(columnas_limpiadas))
print(reporte_limpieza.sort_values("valores_convertidos", ascending=False).head(10))


=== LIMPIEZA COMPLETADA ===
Columnas convertidas a numérico: 281
       variable tipo_original  valores_convertidos  nulos_finales
105    PROM_HNV           str                 1096           1097
140  PCDISC_MEN           str                 1072           1072
179  P15PRI_COM           str                 1072           1072
185  P15SEC_COM           str                 1072           1072
184  P15SEC_COF           str                 1072           1072
183   P15SEC_CO           str                 1072           1072
182  P15SEC_INM           str                 1072           1072
181  P15SEC_INF           str                 1072           1072
180   P15SEC_IN           str                 1072           1072
178  P15PRI_COF           str                 1072           1072


In [53]:
# Parsear columnas numericas a numero
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
categorical_cols = df.select_dtypes(exclude=np.number).columns.tolist()
print(len(numeric_cols), len(categorical_cols))


for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Dropear filas donde GRAPROES es nulo o NAN
df = df.dropna(subset=['GRAPROES'])

281 5


In [54]:
# Sampling 60% train 20% test 20% validation
train_df, temp_df = train_test_split(df, test_size=0.4, random_state=42)
test_df, val_df = train_test_split(temp_df, test_size=0.5, random_state=42)
train_df.head()

,ENTIDAD,NOM_ENT,MUN,NOM_MUN,LOC,NOM_LOC,LONGITUD,LATITUD,ALTITUD,POBTOT,...,VPH_CEL,VPH_INTER,VPH_STVP,VPH_SPMVPI,VPH_CVJ,VPH_SINRTV,VPH_SINLTC,VPH_SINCINT,VPH_SINTIC,TAMLOC
1232,1,Aguascalientes,6,Pabellón de Arteaga,100,San Pedro [Rancho],"102°15'34.559"" W","22°08'54.125"" N",1901.0,16,...,4.0,0.0,2.0,0.0,0.0,0.0,0.0,3.0,0.0,1.0
510,1,Aguascalientes,1,Aguascalientes,2402,San Miguel,"102°11'49.309"" W","21°53'47.597"" N",2020.0,104,...,20.0,5.0,2.0,2.0,1.0,0.0,2.0,15.0,0.0,1.0
1007,1,Aguascalientes,5,Jesús María,52,Mesa de las Carretas (Un Nuevo Amanecer),"102°26'19.748"" W","21°53'22.541"" N",2030.0,17,...,4.0,4.0,3.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
1254,1,Aguascalientes,6,Pabellón de Arteaga,183,El Mezquite [Colonia],"102°13'20.884"" W","22°04'47.345"" N",1969.0,141,...,24.0,4.0,13.0,0.0,0.0,1.0,6.0,25.0,0.0,1.0
913,1,Aguascalientes,3,Calvillo,9998,Localidades de una vivienda,NaN,NaN,NaN,173,...,39.0,9.0,9.0,1.0,1.0,4.0,8.0,37.0,3.0,NaN


In [55]:
# Describe
train_df.describe()

,ENTIDAD,MUN,LOC,ALTITUD,POBTOT,POBFEM,POBMAS,P_0A2,P_0A2_F,P_0A2_M,...,VPH_CEL,VPH_INTER,VPH_STVP,VPH_SPMVPI,VPH_CVJ,VPH_SINRTV,VPH_SINLTC,VPH_SINCINT,VPH_SINTIC,TAMLOC
count,591.0,591.000000,591.000000,568.000000,591.000000,591.000000,591.000000,591.000000,591.000000,591.000000,...,591.000000,591.000000,591.000000,591.000000,591.000000,591.000000,591.000000,591.000000,591.000000,568.000000
mean,1.0,4.556684,728.470389,1928.614437,4276.998308,2189.448393,2087.549915,211.262267,104.472081,106.790186,...,1095.465313,734.365482,536.326565,315.521151,222.500846,17.323181,42.077834,373.737733,4.663283,1.707746
std,0.0,3.281135,1685.357833,114.301240,53201.584186,27332.087832,25869.684441,2473.970601,1221.974517,1252.017023,...,14183.848172,10170.096520,7411.333756,4630.519546,3203.216440,186.627869,400.867898,4085.866136,40.945271,1.397583
min,1.0,0.000000,0.000000,1555.000000,4.000000,0.000000,1.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000
25%,1.0,1.000000,55.000000,1881.000000,19.000000,9.000000,10.000000,1.000000,0.000000,0.000000,...,4.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,3.000000,0.000000,1.000000
50%,1.0,4.000000,212.000000,1931.000000,60.000000,29.000000,30.000000,4.000000,2.000000,2.000000,...,13.000000,4.000000,4.000000,1.000000,1.000000,0.000000,1.000000,9.000000,0.000000,1.000000
75%,1.0,7.000000,494.000000,1999.250000,380.000000,192.000000,195.000000,21.500000,10.000000,10.000000,...,76.000000,29.500000,20.000000,4.000000,4.000000,2.000000,8.000000,50.000000,1.000000,2.000000
max,1.0,11.000000,9999.000000,2512.000000,948990.000000,486917.000000,462073.000000,44372.000000,21893.000000,22479.000000,...,251719.000000,178619.000000,130290.000000,80951.000000,56131.000000,3299.000000,7293.000000,74227.000000,731.000000,13.000000


In [ ]:
TARGET_COL = 'GRAPROES'

X_train = train_df.drop(columns=[TARGET_COL]).copy()
y_train = train_df[TARGET_COL].copy()

X_test = test_df.drop(columns=[TARGET_COL]).copy()
y_test = test_df[TARGET_COL].copy()

In [56]:
# Pipelines especializadas
def column_ratio(X):
    return X[:, [0]] / X[:, [1]]


def ratio_name(function_transformer, feature_names_in):
    return ["ratio"]

def ratio_pipeline():
    return make_pipeline(
        SimpleImputer(strategy="median"),
        FunctionTransformer(column_ratio, feature_names_out=ratio_name),
        StandardScaler(),
    )


log_pipeline = make_pipeline(
    SimpleImputer(strategy="median"),
    FunctionTransformer(np.log, np.exp, feature_names_out="one-to-one"),
    StandardScaler(),
)


default_num_pipeline = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
)

In [ ]:
# Preprocesamiento completo

preprocessing = ColumnTransformer(
    [
        ("default", default_num_pipeline, ["baths", "beds"]),
        ("baths", ratio_pipeline(), ["baths", "sqft"]),
        ("beds", ratio_pipeline(), ["beds", "sqft"]),
        ("log", log_pipeline, ["sqft"])
    ],
    remainder=default_num_pipeline,
)

X_train_prepared = preprocessing.fit_transform(X_train)
if sparse.issparse(X_train_prepared):
    X_train_prepared = X_train_prepared.toarray()
X_train_prepared = pd.DataFrame(
    X_train_prepared,
    columns=preprocessing.get_feature_names_out(),
    index=X_train.index,
)

X_train_prepared.head()

# Configuración de BorutaShap
feature_selector = BorutaShap(
    model=RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1,
        min_samples_leaf=2,
    ),
    importance_measure="shap",
    classification=False,
)

feature_selector.fit(
    X=X_train_prepared,
    y=y_train,
    n_trials=30,
    sample=False,
    train_or_test="test",
    normalize=True,
    verbose=True,
    random_state=42,
)

ValueError: A given column is not a column of the dataframe

In [ ]:
# 3: Identificar columnas numéricas y categóricas
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
categorical_cols = df.select_dtypes(exclude=np.number).columns.tolist()


config = CleanerConfig(
    missing_threshold=0.4,           # Elimina cols con >40% nulos
    fill_numeric_strategy="mean",    # Imputa numéricos con media
    fill_categorical_strategy="unknown",
    remove_outliers=True,
    outlier_method="zscore",         # O "iqr"
    drop_columns=["id", "temp_col"],
    rename_columns={"col_vieja": "col_nueva"},
)
cleaner = DataCleaner(config=config)

df_clean = cleaner.clean(df)

cleaner.save(df_clean, "../data/inegi_limpio.csv")
cleaner.print_report()